In [146]:
from dotenv import load_dotenv
load_dotenv()

from langchain_experimental.text_splitter import SemanticChunker
import pymupdf4llm, os
from langchain_core.documents import Document
from typing import *
from pathlib import Path
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores.azuresearch import AzureSearch 


In [147]:
single_path =r"C:\Users\csc\Desktop\Kuntal_work\PharmaStudyHelper\Data\empa-reg trial summary.pdf"
miultiple_pdf_path = r"C:\Users\csc\Desktop\Kuntal_work\PharmaStudyHelper\Data"

# Single PDF

In [148]:
pages = pymupdf4llm.to_markdown(single_path, page_chunks=True)
pages[0]['metadata']


=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.


{'format': 'PDF 1.5',
 'title': 'EMPA-REG- CV Outcomes Trial Summary',
 'author': 'RHDSST',
 'subject': '',
 'keywords': '',
 'creator': 'Microsoft® Word 2016',
 'producer': 'Microsoft® Word 2016',
 'creationDate': "D:20190522155427-06'00'",
 'modDate': "D:20190522155427-06'00'",
 'trapped': '',
 'encryption': None,
 'file_path': 'C:\\Users\\csc\\Desktop\\Kuntal_work\\PharmaStudyHelper\\Data\\empa-reg trial summary.pdf',
 'page_count': 3,
 'page_number': 1}

In [36]:
def pdf_to_docs(pdf_path: str) -> List[Document]:
    docs = []
    pages = pymupdf4llm.to_markdown(pdf_path, page_chunks=True)
    for p in pages:
        text = p["text"].strip()
        if not text:
            continue
        m = p["metadata"]
        docs.append(Document(
            page_content=text,
            metadata={"source": Path(pdf_path).name, "page": m.get("page_number", ""), "title": m.get("title", "")},
        ))
    return docs

docs = pdf_to_docs(single_path)


=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.


In [37]:
docs

[Document(metadata={'source': 'empa-reg trial summary.pdf', 'page': 1, 'title': 'EMPA-REG- CV Outcomes Trial Summary'}, page_content='**RXFILES TRIAL SUMMARY** APR 2016 –  WWW.RXFILES.CA \n\n# **EMPA-REG: CV Outcomes Trial Summary** \n\n**Empagliflozin: Cardiovascular (CV) Outcomes and Mortality in Patients with Type 2 Diabetes Mellitus (T2DM) In patients with T2DM and at high risk of CV events, does empagliflozin reduce CV risk compared to placebo when added to standard care?** \n\n## **<mark>BOTTOM LINE</mark>**<sup>**1**</sup> \n\n- <mark>Empaglifozin compared to placebo in a high CV risk population:</mark> \n\n   - Benefit: reduced risk of composite cardiovascular events ( <mark>NNT=63/3yrs)</mark> and all cause death ( <mark>NNT=38/3yrs)</mark> \n\n      - The 10mg daily dose provided virtually the same benefit as the 25mg dose. Cost about $300/100days. \n\n   - Benefit realized despite A1C not reaching target (A1C=7.8%); mean change was approximately \uf0a3 0.6% \n\n   - `o` Harm

# Multiple PDF

In [47]:
def PdfToMarkDown(pdf_dir: str)-> List[Document]:
    docs=[]
    for pdf in sorted(Path(pdf_dir).glob("*.pdf")):
        pages=pymupdf4llm.to_markdown(str(pdf), page_chunks=True)
        for p in pages:
            text = p['text'].strip()
            if not text: 
                continue 
            docs.append(Document(
                page_content=text, 
                metadata = {'source': pdf.name, "page": p['metadata'].get("page_number")}
            ))

    return docs 
docs=PdfToMarkDown(miultiple_pdf_path)





=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.
OCR on page.number=4/5.
OCR on page.number=5/6.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.
OCR on page.number=4/5.
OCR on page.number=5/6.

=== Document parser messages ===
Using Tesseract for OCR processing.


In [82]:
embeddings = AzureOpenAIEmbeddings(
    model=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

In [86]:
single_pdf_only=r"C:\Users\csc\Desktop\Kuntal_work\PharmaStudyHelper\Data\OneData"
docs=pdf_to_docs(single_path)


=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.


In [87]:
chunker = SemanticChunker(embeddings=embeddings, breakpoint_threshold_type='percentile')

In [89]:
chunks=chunker.split_documents(documents=docs)

In [93]:
print(len(docs))
print(len(chunks))
print(chunks[0].page_content[:300])
print(chunks[0].metadata)

3
10
**RXFILES TRIAL SUMMARY** APR 2016 –  WWW.RXFILES.CA 

# **EMPA-REG: CV Outcomes Trial Summary** 

**Empagliflozin: Cardiovascular (CV) Outcomes and Mortality in Patients with Type 2 Diabetes Mellitus (T2DM) In patients with T2DM and at high risk of CV events, does empagliflozin reduce CV risk compa
{'source': 'empa-reg trial summary.pdf', 'page': 1, 'title': 'EMPA-REG- CV Outcomes Trial Summary'}


In [94]:
vector_store = AzureSearch(
    azure_search_endpoint=os.getenv("AZURE_SEARCH_ENDPOINT"),
    azure_search_key=os.getenv("AZURE_SEARCH_API_KEY"),
    index_name=os.getenv("AZURE_SEARCH_INDEX_NAME"),
    embedding_function=embeddings.embed_query,
)

In [98]:
vector_store.add_documents(chunks)

['NDU2OTU4ZmQtZjc4YS00YTBkLWJmNDUtMjU5NjhjMDI3YjBi',
 'MGYxNGUyNDQtOGQ0ZC00OGYyLWI3YjktMTE4YThjMGY1OGZl',
 'ZmYxOTM4ZGQtYTJlMi00NjEyLWExYmMtZTk2MThkYmNhNDc4',
 'MTgwMmY0MTctZGNhMC00MmMxLWE4NDMtNmEwMWNlMGQwYTEx',
 'NjFjOTI3ODYtNjk2ZC00MGY3LWI3ZDktODEwNzM0MWE5Yzg0',
 'YmYxMjM0NGItMDU4ZS00Zjc3LWJhZWEtMzg5NDA4MGU1NmJh',
 'ZmMwYzU1OWEtNzQ1ZS00MzFlLWFkYzQtYjdiODI2MDlmOTc5',
 'NTExNzIwYjktOTNhOC00MTFlLWJlZjUtYjA1YjA1NjI4ZGQ0',
 'N2I4NTcwNmUtY2ZiMy00N2EzLTk0MzUtMDA1NjI1ZTlmYzY1',
 'M2E2Mjc1NmEtNDA2ZC00ZTkzLTk1NmYtZGVhMWQyMDBhNmRl']

In [100]:
embeddings.embed_query("hi")

[-0.003753662109375,
 -0.0191802978515625,
 0.01221466064453125,
 0.03277587890625,
 0.0161895751953125,
 -0.037261962890625,
 -0.028228759765625,
 0.06500244140625,
 0.0009646415710449219,
 -0.057525634765625,
 0.0014333724975585938,
 -0.03265380859375,
 -0.0307464599609375,
 0.004779815673828125,
 0.032928466796875,
 0.0192413330078125,
 -0.04193115234375,
 -0.0029048919677734375,
 0.024749755859375,
 0.046630859375,
 0.03778076171875,
 0.034149169921875,
 -0.00243377685546875,
 0.0362548828125,
 0.0163421630859375,
 0.0018634796142578125,
 0.0021209716796875,
 -0.00887298583984375,
 0.032073974609375,
 -0.026336669921875,
 0.0013456344604492188,
 -0.037811279296875,
 0.02581787109375,
 -0.037017822265625,
 -0.01654052734375,
 -0.0126495361328125,
 -0.025604248046875,
 0.033050537109375,
 0.0155181884765625,
 -0.03778076171875,
 0.0249786376953125,
 -0.004978179931640625,
 0.046478271484375,
 0.015869140625,
 -0.019805908203125,
 0.015655517578125,
 -0.0269775390625,
 0.0164642333984

In [ ]:
vectore_store=AzureSearch(
    embedding_function=embeddings.embed_query, 
    azure_search_endpoint=os.getenv("azure_search_endpoint".upper()),
    azure_search_key=os.getenv("azure_search_key".upper()),
    index_name='teri_ma_ki_chut'
)


vectore_store.add_documents(chunks)

In [116]:
retriever = vector_store.as_retriever(
    search_type="hybrid",
    k=4,
)

In [119]:
q='what is the cardio vascular outcome of this study?'

from langchain_core.vectorstores.utils import maximal_marginal_relevance as mmr 

In [121]:
q_embd=vectore_store.embed_query(q)

candi=vectore_store.similarity_search(q, k=20)
candi

[Document(metadata={'id': 'MmFhNDk4YjYtNDllZi00OWE5LTg3YjQtMTc3ZTdmMTYwNWE5', 'source': 'empa-reg trial summary.pdf', 'page': 2, 'title': 'EMPA-REG- CV Outcomes Trial Summary'}, page_content='**RXFILES TRIAL SUMMARY** APR 2016 –  WWW.RXFILES.CA \n\n|**TABLE 2: ADVERSEEVENTS(AE)**|\n|---|\n\n\n\n|**CLINICALENDPOINTS**|**EMPAGLIFLOZIN**<br>**POOLED**|**EMPAGL**|**IFLOZIN**|**PLACEBO**|**P VALUE**|**ARR/ARI**<br>|**NNT/NNH**<br>|**COMMENTS**|\n|---|---|---|---|---|---|---|---|---|\n||n=4687|**10MG**n=2345|**25MG**n=2342|**n=2333**||**(POOLED) **|**/3.1YRS**||\n|Severe AE|23.5%(n=1100)|22.9%(n=536)|24.1%(n=564)|25.4%(n=592)|<0.05|↓1.9%|53|Proportion of patients with confirmed|\n|Serious AE(Any)|38.2%(n=1789)|37.4%(n=876)|39.0%(n=913)|42.3%(n=988)|<0.001|↓4.1%|24|hypoglycemic events, acute renal<br>|\n|Serious AE(Death)|3.8%(n=176)|4.1%(n=97)|3.4%(n=79)|5.1%(n=119)|<0.01|↓1.3%|77|failure, diabetic ketoacidosis,<br>thromboembolic events bone|\n|AE leading to D/C of<br>studydrug|17.3% (n=813)

In [124]:
candi_embedding = embeddings.embed_documents([d.page_content for d in candi])

In [143]:
import numpy as np
idx=mmr(np.array(q_embd), candi_embedding, lambda_mult=0.5, k=5)
idx

[1, 9, 3, 5, 8]

In [144]:
final=[candi[i] for i in idx]
for x in final:
    print(x.page_content)

2016 Apr 25. Verma S, Mazer CD, Al-Omran M, et al. Cardiovascular Outcomes and Safety of Empagliflozin in Patients With Type 2 Diabetes Mellitus and **Peripheral Artery Disease** : 

A Subanalysis of **EMPA-REG OUTCOME** . Circulation.
2015 Sep 17.
**RXFILES TRIAL SUMMARY** APR 2016 –  WWW.RXFILES.CA 

# **EMPA-REG: CV Outcomes Trial Summary** 

**Empagliflozin: Cardiovascular (CV) Outcomes and Mortality in Patients with Type 2 Diabetes Mellitus (T2DM) In patients with T2DM and at high risk of CV events, does empagliflozin reduce CV risk compared to placebo when added to standard care?** 

## **<mark>BOTTOM LINE</mark>**<sup>**1**</sup> 

- <mark>Empaglifozin compared to placebo in a high CV risk population:</mark> 

   - Benefit: reduced risk of composite cardiovascular events ( <mark>NNT=63/3yrs)</mark> and all cause death ( <mark>NNT=38/3yrs)</mark> 

      - The 10mg daily dose provided virtually the same benefit as the 25mg dose. Cost about $300/100days.
2018;137:119–129. doi: 10.

# Azure Document Intelligence

> This same like the PYmupdf4llm --> can do ocr, keep the headings and retur the markdown output also hold the table format etc 

In [169]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient

def AzureIntelMarkdownCreator(path):
# client 
    di_client = DocumentIntelligenceClient(
        endpoint=os.getenv("AZURE_DOCINTEL_ENDPOINT"),
        credential=AzureKeyCredential(os.getenv("AZURE_DOCINTEL_KEY")))


    with open(path, 'rb') as f:
        poller = di_client.begin_analyze_document(
            "prebuilt-layout",
            body=f,
            content_type='application/octet-stream',
            output_content_format='markdown'
        )
    result = poller.result()

    docs=[]


    for page in result.pages:
        text = "".join(
            result.content[s.offset : s.offset + s.length] for s in page.spans
        ).strip()
        if not text:
            continue
        docs.append(Document(
            page_content=text, 
            metadata={"page": page.page_number}
        ))

    print(len(docs) ,"pages")
    return docs

docs=AzureIntelMarkdownCreator(path=single_path)


2 pages


In [170]:
from pprint import pprint

for x in docs:
    print(x.metadata)
    print(x.page_content[:300])

{'page': 1}
<!-- PageHeader="RXFILES TRIAL SUMMARY" -->
<!-- PageHeader="APR 2016 - www.RxFILES.CA" -->

EMPA-REG: CV Outcomes Trial Summary
Empagliflozin: Cardiovascular (CV) Outcomes and Mortality in Patients with Type 2 Diabetes Mellitus (T2DM)
In patients with T2DM and at high risk of CV events, does empagl
{'page': 2}
<!-- PageHeader="RXFILES TRIAL SUMMARY" -->
<!-- PageHeader="APR 2016 - www.RxFILES.CA" -->


<table>
<tr>
<th colspan="9">TABLE 2: ADVERSE EVENTS (AE)</th>
</tr>
<tr>
<th rowspan="2">CLINICAL ENDPOINTS</th>
<th rowspan="2">EMPAGLIFLOZIN POOLED n=4687</th>
<th colspan="2">EMPAGLIFLOZIN</th>
<th rows


{'page': 1}